In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [14]:
import pandas as pd
import os

# lecture des différents datasets, celui de 2002 en txt et celui de 2007 à 2022 en csv
data_dir = 'data'
file_txt_2002 = os.path.join(data_dir, 'PR02_BVot_T1T2.txt')
file_csv_cumul = os.path.join(data_dir, 'resultats_elections_cumul_presidentielles_rennes.csv')

# Extraction des données de la commune de Rennes uniquement sur le dataset de 2002
print("Extraction de Rennes depuis le fichier national 2002...")

cols_02 = [
    'NUMERO_TOUR', 'CODE_DEPT', 'CODE_COMMUNE', 'NOM_COMMUNE', 'NUMERO_BV',
    'NB_INSCRITS', 'NB_VOTANTS', 'NB_EXPRIMES', 'NUM_DEPOT', 'NOM_CANDIDAT',
    'PRENOM_CANDIDAT', 'SIGLE', 'NB_VOIX'
]

chunks = pd.read_csv(file_txt_2002, sep=';', header=None, skiprows=17, 
                     names=cols_02, encoding='latin-1', chunksize=100000,
                     dtype={'CODE_DEPT': str, 'CODE_COMMUNE': str, 'NUMERO_BV': str})

df_02_rennes = pd.concat([chunk[(chunk['CODE_DEPT'] == '35') & (chunk['CODE_COMMUNE'] == '238')] for chunk in chunks])

df_02_final = pd.DataFrame({
    'CODE_ELECTION': 'P02',
    'DATE_ELECTION': 2002,
    'NUMERO_TOUR': df_02_rennes['NUMERO_TOUR'],
    'NUMERO_BV': df_02_rennes['NUMERO_BV'].str.lstrip('0'),
    'NB_INSCRITS': df_02_rennes['NB_INSCRITS'],
    'NB_EXPRIMES': df_02_rennes['NB_EXPRIMES'],
    'NOM_CANDIDAT': df_02_rennes['NOM_CANDIDAT'].str.strip().str.upper(),
    'NB_VOIX': df_02_rennes['NB_VOIX']
})

# Transformation du dataset de 2007 à 2022 en lecture sous forme de ligne 
print("Transformation du fichier cumulé Rennes (2007-2017)...")
df_cumul = pd.read_csv(file_csv_cumul, sep=';')

id_vars = ['CODE_ELECTION', 'DATE_ELECTION', 'NUMERO_TOUR', 'NUMERO_LIEU', 'NB_INSCRITS', 'NB_EXPRIMES']
list_parts = []

for i in range(1, 13):
    c_nom = f'CANDIDAT_{i}'
    c_voix = f'NB_VOIX_{i}'
    if c_nom in df_cumul.columns:
        temp = df_cumul[id_vars + [c_nom, c_voix]].copy()
        temp.columns = id_vars + ['NOM_CANDIDAT', 'NB_VOIX']
        list_parts.append(temp)

df_cumul_long = pd.concat(list_parts)
df_cumul_long['NUMERO_BV'] = df_cumul_long['NUMERO_LIEU'].astype(str).str.lstrip('0')


df_cumul_long['NOM_CANDIDAT'] = df_cumul_long['NOM_CANDIDAT'].str.strip().str.upper()
df_cumul_long = df_cumul_long.drop(columns=['NUMERO_LIEU'])

# Fusion des deux datasets et sauvegarde sur resultat_election_presidentiel_complet 
df_master = pd.concat([df_02_final, df_cumul_long], ignore_index=True)
df_master = df_master.dropna(subset=['NOM_CANDIDAT'])

output_path = os.path.join(data_dir, 'resultat_election_presidentiel_complet.csv')
df_master.to_csv(output_path, index=False, sep=';')

print(f"Terminé ! Fichier créé : {output_path}")

Extraction de Rennes depuis le fichier national 2002...
Transformation du fichier cumulé Rennes (2007-2017)...
Terminé ! Fichier créé : data\resultat_election_presidentiel_complet.csv


In [15]:
df_master = pd.read_csv('data/resultat_election_presidentiel_complet.csv', sep=';')

print(f"Total : {len(df_master)} lignes")
display(df_master.groupby('DATE_ELECTION')['NB_VOIX'].sum()) # Vérif des voix par année
display(df_master.sample(5))

Total : 11926 lignes


DATE_ELECTION
2002          163539.0
2007-04-22    833482.0
2007-05-06    788139.0
2007-06-05    112408.0
2012-04-22    806698.0
2012-05-06    664679.0
2012-06-05    120845.0
2017-04-23    848675.0
2017-05-07    613052.0
2017-07-05    120957.0
2022-04-10    370821.0
2022-04-24    664067.0
2022-09-04    367936.0
Name: NB_VOIX, dtype: float64

,CODE_ELECTION,DATE_ELECTION,NUMERO_TOUR,NUMERO_BV,NB_INSCRITS,NB_EXPRIMES,NOM_CANDIDAT,NB_VOIX
1391,P02,2002,1,303.0,937,624,BESANCENOT,37.0
8817,P07,2007-04-22,1,121.0,906,802,ROYAL SÉGOLÈNE S,181.0
5630,P12,2012-04-22,1,NaN,3181,2686,SARKOZY NICOLAS,672.0
6012,P17,2017-04-23,1,44.0,3834,3013,HAMON BENOÎT,477.0
3872,P12,2012-04-22,1,211.0,1010,785,LE PEN MARINE,85.0


In [17]:
# 1. Le dictionnaire de correspondance (Mapping)
blocs = {
    'EXG': ['LAGUILLER', 'BESANCENOT', 'GLUCKSTEIN', 'SCHIVARDI', 'ARTHAUD', 'POUTOU'],
    'GCH': ['JOSPIN', 'HUE', 'MAMÈRE', 'CHEVÈNEMENT', 'TAUBIRA', 'ROYAL', 'BUFFET', 
            'VOYNET', 'BOVÉ', 'HOLLANDE', 'MÉLENCHON', 'JOLY', 'HAMON', 'HIDALGO', 'ROUSSEL', 'JADOT'],
    'CTR': ['BAYROU', 'MADELIN', 'MACRON', 'LEPAGE'],
    'DRT': ['CHIRAC', 'BOUTIN', 'SARKOZY', 'FILLON', 'PÉCRESSE'],
    'EXD': ['LE PEN', 'MÉGRET', 'ZEMMOUR'],
    'DIV': ['SAINT-JOSSE', 'NIHOUS', 'DE VILLIERS', 'DUPONT-AIGNAN', 'CHEMINADE', 'LASSALLE', 'ASSELINEAU']
}

# 2. Inversion du dictionnaire pour le mapping rapide
name_to_bloc = {name: bloc for bloc, names in blocs.items() for name in names}

# 3. Application sur le DataFrame
def get_bloc(nom):
    # On cherche si un nom du dictionnaire est présent dans la cellule
    for ref_name in name_to_bloc:
        if ref_name in str(nom).upper():
            return name_to_bloc[ref_name]
    return 'AUTRE'

df_master['BLOC_POLITIQUE'] = df_master['NOM_CANDIDAT'].apply(get_bloc)

# 4. Affichage du résultat final
print("Aperçu du mapping par Bloc :")
display(df_master.sample(25))

# 5. Sauvegarde finale
df_master.to_csv('data/master_rennes_with_bloc.csv', index=False, sep=';')

Aperçu du mapping par Bloc :


,CODE_ELECTION,DATE_ELECTION,NUMERO_TOUR,NUMERO_BV,NB_INSCRITS,NB_EXPRIMES,NOM_CANDIDAT,NB_VOIX,BLOC_POLITIQUE
11195,P17,2017-04-23,1,241.0,1223,987,FILLON FRANÇOIS,247.0,DRT
5345,P17,2017-04-23,1,443.0,1230,925,MACRON EMMANUEL,280.0,CTR
7502,P07,2007-04-22,1,16.0,1413,1232,VOYNET DOMINIQUE D,24.0,GCH
7846,P07,2007-04-22,1,85.0,1010,821,VOYNET DOMINIQUE D,9.0,GCH
3656,P07,2007-05-06,2,8.0,7097,5676,ROYAL SÉGOLÈNE S,3943.0,GCH
6021,P07,2007-04-22,1,334.0,1229,1063,BAYROU FRANÇOIS F,259.0,CTR
6150,P12,2012-04-22,1,162.0,990,792,MÉLENCHON JEAN-LUC,132.0,GCH
2863,P07,2007-05-06,2,271.0,1112,874,SARKOZY NICOLAS N,297.0,DRT
5802,P22,2022-04-10,1,453.0,1137,894,LASSALLE JEAN,11.0,DIV
1238,P02,2002,1,272.0,1117,743,TAUBIRA,21.0,GCH
